In [21]:
import numpy as np
import pandas as pd

# =====================
# Plotting
# =====================
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
import seaborn as sns

# =====================
# SciPy / Stats
# =====================
from scipy.stats import norm
import statsmodels.api as sm
import statsmodels.formula.api as smf

# =====================
# sklearn – model selection
# =====================
from sklearn.model_selection import train_test_split

# =====================
# sklearn – models
# =====================
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# =====================
# sklearn – metrics
# =====================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    r2_score,
    mean_squared_error,
)

# =====================
# Utilities
# =====================
from itertools import combinations, product
from tqdm import tqdm

# Some auxiliary functions

In [22]:
# Construct the replicator model
def construct_replicator_features(X):
    linear_part = X.values  # shape: [n_samples, 11]

    # Cross-species interactions only (i < j)
    cross_terms = []
    for i, j in combinations(range(X.shape[1]), 2):
        interaction = - (X.iloc[:, i] * X.iloc[:, j]).values.reshape(-1, 1)
        cross_terms.append(interaction)

    quadratic_part = np.hstack(cross_terms)  # shape: [n_samples, 55]
    features = np.hstack([linear_part, quadratic_part])  # shape: [n_samples, 66]
    return features

# Take the mid point of each Nugent score class. Easier to multiply it by 10 here and divide later
def convert_to_levels(y):
    return pd.cut(
        y,
        bins=[-0.1, 3, 6, 10],  # 3 categories
        labels=[15, 50, 85]     # Midpoints
    ).astype(int)

# Nugent Score transformation
def transform_y(y, N0=None, c=None):
    if N0 is None or c is None:
        return y
    N_max = 10
    d = N_max - 2 * N0 + c
    return np.log(N0 + d) - np.log(N_max - y + c)
    
import matplotlib.pyplot as plt


#Summarize metrics
def summarize_metrics(metrics_dict, name):
    print(f"\n{name} Average Metrics over {N} runs:")
    for key, values in metrics_dict.items():
        print(f"{key.capitalize():<10}: {np.mean(values):.4f} ± {np.std(values):.4f}")

# Importing the data

In [23]:
import pandas as pd

data = pd.read_csv(
    "14_taxa_data.csv"
)

# Response and raw taxonomic abundances
y = data.iloc[:, 3].astype(int)
X_raw = data.iloc[:, 4:]


def group_phyla_separating_lactobacilli(X):
    """
    Group taxonomic abundance columns at the phylum level, while keeping
    four Lactobacillus species as separate features.

    Outputs:
        Lactobacillus_iners
        Lactobacillus_crispatus
        Lactobacillus_jensenii
        Lactobacillus_gasseri
        Bacillota_I_other
        all other phyla
    """

    separated_lactobacilli = {
        "lactobacillus iners": "Lactobacillus_iners",
        "lactobacillus crispatus": "Lactobacillus_crispatus",
        "lactobacillus jensenii": "Lactobacillus_jensenii",
        "lactobacillus gasseri": "Lactobacillus_gasseri",
    }

    grouped_columns = {}

    for column in X.columns:
        taxonomy = str(column).split(";")

        # Extract names without prefixes such as p__ and s__
        ranks = {}

        for component in taxonomy:
            component = component.strip()

            if "__" in component:
                prefix, name = component.split("__", 1)
                ranks[prefix] = name.strip()

        phylum = ranks.get("p")
        species = ranks.get("s")

        # Skip features without a valid phylum assignment
        if not phylum or phylum.lower() in {
            "unassigned",
            "unclassified",
            "unknown",
        }:
            continue

        species_normalized = species.lower() if species else None

        # Keep the four selected Lactobacillus species separate
        if species_normalized in separated_lactobacilli:
            new_name = separated_lactobacilli[species_normalized]

        # Group the remainder of Bacillota_I
        elif phylum == "Bacillota_I":
            new_name = "Bacillota_I_other"

        # Group all other taxa at the phylum level
        else:
            new_name = phylum

        grouped_columns.setdefault(new_name, []).append(column)

    # Sum all original columns assigned to each new feature
    X_grouped = pd.DataFrame(
        {
            group_name: X[columns].sum(axis=1)
            for group_name, columns in grouped_columns.items()
        },
        index=X.index,
    )

    return X_grouped


X_grouped = group_phyla_separating_lactobacilli(X_raw)

feature_names = [
    "Actinomycetota",
    "Bacillota_A_368345",
    "Bacillota_C",
    "Bacillota_I_other",
    "Lactobacillus_iners",
    "Lactobacillus_crispatus",
    "Lactobacillus_jensenii",
    "Lactobacillus_gasseri",
    "Bacteroidota",
    "Campylobacterota_A",
    "Fusobacteriota",
    "Patescibacteria",
    "Pseudomonadota",
    "Synergistota",
]

missing = [
    name
    for name in feature_names
    if name not in X_grouped.columns
]

unexpected = [
    name
    for name in X_grouped.columns
    if name not in feature_names
]

if missing:
    raise ValueError(
        f"Expected features missing from grouped data: {missing}"
    )

if unexpected:
    print(
        "Grouped features not included in feature_names: "
        f"{unexpected}"
    )

X_grouped = X_grouped[feature_names]

row_totals = X_grouped.sum(axis=1)

X_freq = X_grouped.div(
    row_totals.replace(0, pd.NA),
    axis=0,
)

print(X_freq.columns.tolist())

['Actinomycetota', 'Bacillota_A_368345', 'Bacillota_C', 'Bacillota_I_other', 'Lactobacillus_iners', 'Lactobacillus_crispatus', 'Lactobacillus_jensenii', 'Lactobacillus_gasseri', 'Bacteroidota', 'Campylobacterota_A', 'Fusobacteriota', 'Patescibacteria', 'Pseudomonadota', 'Synergistota']


In [24]:
X_freq

,Actinomycetota,Bacillota_A_368345,Bacillota_C,Bacillota_I_other,Lactobacillus_iners,Lactobacillus_crispatus,Lactobacillus_jensenii,Lactobacillus_gasseri,Bacteroidota,Campylobacterota_A,Fusobacteriota,Patescibacteria,Pseudomonadota,Synergistota
0,0.000000,0.000880,0.000000,0.000000,0.085312,0.913808,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0
1,0.000000,0.000000,0.000000,0.002974,0.000000,0.000000,0.101115,0.895911,0.000000,0.0,0.000000,0.0,0.000000,0.0
2,0.003745,0.001498,0.000000,0.001498,0.747566,0.000000,0.245693,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0
3,0.000000,0.001360,0.000000,0.000000,0.000000,0.998640,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0
4,0.058221,0.039028,0.336532,0.031990,0.055662,0.001280,0.000000,0.000000,0.292386,0.0,0.184901,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,0.085997,0.003486,0.004067,0.065078,0.764672,0.000000,0.000000,0.000000,0.076700,0.0,0.000000,0.0,0.000000,0.0
390,0.000000,0.000000,0.000000,0.007922,0.766606,0.000000,0.216332,0.000000,0.009141,0.0,0.000000,0.0,0.000000,0.0
391,0.000000,0.001330,0.000000,0.615027,0.000000,0.375665,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.007979,0.0
392,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0


# Calibrating the Ridge regression parameters (alpha fixed at 0.05)

In [5]:
def run_multiple_evaluations_with_transformed_regression(X, y, N0=None, c=None, N=100, test_size=0.2, random_state_seed=42, alpha=0.05):
    unique_labels = sorted(y.unique())
    metrics_reg = {'accuracy': [], 'precision': [] , 'f1': []}

    for i in range(N):
        X_train, X_test, y_train_raw, y_test_raw = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # Transform y_train and y_test
        y_train_transformed = transform_y(y_train_raw/10, N0, c)
        y_test_transformed = transform_y(y_test_raw/10, N0, c)

        # Replicator features
        X_train_rep = construct_replicator_features(X_train)
        X_test_rep = construct_replicator_features(X_test)

        # Linear regression
        #No intercept
        X_train_sm = X_train_rep
        X_test_sm = X_test_rep

        model = Ridge(alpha=alpha, fit_intercept=False)
        model.fit(X_train_sm, y_train_transformed)


        # Predict and inverse-transform
        y_pred_transformed = model.predict(X_test_sm)

        y_pred_inverse = []
        if N0 is not None and c is not None:
            N_max = y.max()
            #print(N_max)
            for pred in y_pred_transformed:
                pred = max(min(pred, 100), -100)
                N_pred = N_max + c - (N_max - N0 + c) / np.exp(pred)
                closest_label = min(unique_labels, key=lambda x: abs(x - 10 * N_pred))
                y_pred_inverse.append(closest_label)
        else:
            # No transformation, so predict directly and round
            y_pred_inverse = np.round(10*y_pred_transformed).astype(int)
            y_pred_inverse = np.clip(y_pred_inverse, min(unique_labels), max(unique_labels))

        y_pred_inverse = np.array(y_pred_inverse)

        # Metrics
        metrics_reg['accuracy'].append(accuracy_score(y_test_raw, y_pred_inverse))
        metrics_reg['precision'].append(precision_score(y_test_raw, y_pred_inverse, average='weighted', zero_division=0))
        metrics_reg['f1'].append(f1_score(y_test_raw, y_pred_inverse, average='weighted', zero_division=0))

    return metrics_reg

In [378]:
#Repeat the process 100 times to prevent bias
N = 100
results = []

# Parameter sweep
N0_values = np.arange(1, 9, 0.5)
c_values = np.arange(0.5, 5.5, 0.5)


# Apply label conversion for 3 classes
y_levels = convert_to_levels(y)
  
for N0 in N0_values:
    for c in c_values:
        metrics_reg = run_multiple_evaluations_with_transformed_regression(X_freq, y_levels, N0=N0, c=c, N=N, alpha=0.05)
        results.append({
            'N0': N0,
            'c': c,
            'regression_accuracy': np.mean(metrics_reg['accuracy']),
            'regression_precision': np.mean(metrics_reg['precision']),
            'regression_f1': np.mean(metrics_reg['f1']),
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Display results
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(results_df)

      N0    c  regression_accuracy  regression_precision  regression_f1
0    1.0  0.5             0.268861              0.551018       0.166689
1    1.0  1.0             0.268101              0.544455       0.165445
2    1.0  1.5             0.268354              0.545177       0.166059
3    1.0  2.0             0.269367              0.558700       0.168384
4    1.0  2.5             0.270633              0.573919       0.171320
5    1.0  3.0             0.272278              0.582797       0.174704
6    1.0  3.5             0.275063              0.606176       0.180661
7    1.0  4.0             0.278101              0.628081       0.186739
8    1.0  4.5             0.280759              0.641269       0.191745
9    1.0  5.0             0.285316              0.644857       0.200530
10   1.5  0.5             0.491266              0.713681       0.483535
11   1.5  1.0             0.492911              0.716239       0.486807
12   1.5  1.5             0.495443              0.717766       0

# Ridge regression and fitting

In [6]:
def fit_transformed_ridge_on_full_data(X, y, N0=None, c=None, alpha=1.0, n_boot=1000, random_state=None):
    if random_state:
        np.random.seed(random_state)
    
    # Transform y
    y_transformed = transform_y(y / 10, N0, c)

    # Replicator features
    X_rep = construct_replicator_features(X)

    # Build feature names
    base_names =  [
    "Actinomycetota",
    "Bacillota_A_368345",
    "Bacillota_C",
    "Bacillota_I_other",
    "Lactobacillus_iners",
    "Lactobacillus_crispatus",
    "Lactobacillus_jensenii",
    "Lactobacillus_gasseri",
    "Bacteroidota",
    "Campylobacterota_A",
    "Fusobacteriota",
    "Patescibacteria",
    "Pseudomonadota",
    "Synergistota",
]
    if X.shape[1] > len(base_names):
        base_names += [f"x{i}" for i in range(len(base_names), X.shape[1])]

    interaction_names = [
        f"{base_names[i]} * {base_names[j]}"
        for i, j in combinations(range(X.shape[1]), 2)
    ]
    feature_names = base_names + interaction_names

    # Fit Ridge regression
    model = Ridge(alpha=alpha, fit_intercept=False)
    model.fit(X_rep, y_transformed)
    r2 = model.score(X_rep, y_transformed)

    # Collect coefficients
    model_params_df = pd.DataFrame([model.coef_], columns=feature_names)


    return model_params_df, model, r2

def fit_transformed_ridge_on_full_data(
    X, y, N0=None, c=None, alpha=1.0, n_boot=1000, random_state=None
):
    if random_state is not None:
        np.random.seed(random_state)

    # Ensure arrays for internal use (but keep original types for nicer indexing if you want)
    X_is_df = isinstance(X, pd.DataFrame)
    y_is_series = isinstance(y, pd.Series)

    # Transform y
    y_transformed = transform_y((y / 10), N0, c)

    # Replicator features (apply to full X)
    X_rep = construct_replicator_features(X)

    # Build feature names
    base_names = [
    "Actinomycetota",
    "Bacillota_A_368345",
    "Bacillota_C",
    "Bacillota_I_other",
    "Lactobacillus_iners",
    "Lactobacillus_crispatus",
    "Lactobacillus_jensenii",
    "Lactobacillus_gasseri",
    "Bacteroidota",
    "Campylobacterota_A",
    "Fusobacteriota",
    "Patescibacteria",
    "Pseudomonadota",
    "Synergistota",
]
    if X.shape[1] > len(base_names):
        base_names += [f"x{i}" for i in range(len(base_names), X.shape[1])]

    interaction_names = [
        f"{base_names[i]} * {base_names[j]}"
        for i, j in combinations(range(X.shape[1]), 2)
    ]
    feature_names = base_names + interaction_names

    # Fit Ridge on full data
    model = Ridge(alpha=alpha, fit_intercept=False)
    model.fit(X_rep, y_transformed)
    r2 = model.score(X_rep, y_transformed)
    coef_full = model.coef_.copy()

    # Prepare bootstrap storage
    n_samples = X.shape[0]
    p = len(feature_names)
    boot_coefs = np.zeros((n_boot, p))

    # Bootstrap loop (use .iloc for DataFrame row sampling)
    for b in range(n_boot):
        idx = np.random.choice(n_samples, n_samples, replace=True)
        Xb = X.iloc[idx] if X_is_df else X[idx]
        yb = y.iloc[idx] if y_is_series else y[idx]

        yb_transformed = transform_y((yb / 10), N0, c)
        Xb_rep = construct_replicator_features(Xb)

        boot_model = Ridge(alpha=alpha, fit_intercept=False)
        boot_model.fit(Xb_rep, yb_transformed)
        boot_coefs[b, :] = boot_model.coef_

    # Bootstrap summaries
    boot_mean = np.mean(boot_coefs, axis=0)
    boot_std = np.std(boot_coefs, axis=0, ddof=1)

    # 95% percentile confidence intervals
    ci_lower = np.percentile(boot_coefs, 2.5, axis=0)
    ci_upper = np.percentile(boot_coefs, 97.5, axis=0)


    # Assemble results DataFrame
    results_df = pd.DataFrame({
        "coef": coef_full,
        "boot_mean": boot_mean,
        "boot_std": boot_std,
        "ci_lower_95": ci_lower,
        "ci_upper_95": ci_upper
    }, index=feature_names)

    return results_df, model, r2

Getting the coefficients with the full dataset:

In [7]:
y_levels = convert_to_levels(y)

N0 = 4.5
c = 4.0

results_df, model, r2 = fit_transformed_ridge_on_full_data(
    X_freq, y_levels, N0=N0, c=c, alpha = 0.05
)

print("Estimated Parameters:\n", results_df['coef'].T)

Estimated Parameters:
 Actinomycetota                      0.660523
Bacillota_A_368345                  0.289821
Bacillota_C                         0.551550
Bacillota_I_other                  -0.090081
Lactobacillus_iners                -0.214750
                                      ...   
Fusobacteriota * Pseudomonadota     0.010765
Fusobacteriota * Synergistota       0.002224
Patescibacteria * Pseudomonadota    0.000280
Patescibacteria * Synergistota     -0.000003
Pseudomonadota * Synergistota       0.000802
Name: coef, Length: 105, dtype: float64


# Monte Carlo cross validation Performance results

In [8]:
def run_multiple_evaluations_with_ridge(X, y, N0=None, c=None, N=100, test_size=0.2, alpha = 0.5, random_state_seed=42):

    unique_labels = sorted(y.unique())
    metrics_reg = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    model_params_list = []
    results = []

    for i in range(N):
        # X_train, X_test, y_train_raw, y_test_raw = train_test_split(
        #     X, y, test_size=test_size, random_state=random_state_seed + i
        # )

        # Ensure consistent splits regardless of columns
        train_idx, test_idx = train_test_split(
            X.index, test_size=test_size, random_state=random_state_seed + i
        )
        # if i==0:
        #     print(train_idx)
        
        # Subset using indices
        X_train = X.loc[train_idx]
        X_test = X.loc[test_idx]
        y_train_raw = y.loc[train_idx]
        y_test_raw = y.loc[test_idx]

        # Transform y
        y_train_transformed = transform_y(y_train_raw/10, N0, c)
        y_test_transformed = transform_y(y_test_raw/10, N0, c)

        # Replicator features
        X_train_rep = construct_replicator_features(X_train)
        X_test_rep = construct_replicator_features(X_test)

        # Ridge regression (no intercept)
        if alpha==0:
            model = sm.OLS(y_train_transformed, X_train_rep).fit()
            r20 = model.rsquared
            residuals0 = model.resid
            mse0 = np.mean(residuals0**2)
        else:
            model = Ridge(alpha=alpha, fit_intercept=False)
            model.fit(X_train_rep, y_train_transformed)
            model_params_list.append(model.coef_)

        #print(model.coef_)
        # Predict and inverse-transform
        y_pred_transformed = model.predict(X_test_rep)
        y_pred_inverse = []
        # if i==0:
        #         print(model.coef_)
        
        if N0 is not None and c is not None:
            N_max = 10
            for pred in y_pred_transformed:
                pred = max(min(pred, 100), -100)
                N_pred = N_max + c - (N_max - N0 + c) / np.exp(pred)
                closest_label = min(unique_labels, key=lambda x: abs(x - 10 * N_pred))
                y_pred_inverse.append(closest_label)
        else:
            y_pred_inverse = np.round(10*y_pred_transformed).astype(int)
            y_pred_inverse = np.clip(y_pred_inverse, min(unique_labels), max(unique_labels))

        y_pred_inverse = np.array(y_pred_inverse)

        # Compute metrics
        metrics_reg['accuracy'].append(accuracy_score(y_test_raw, y_pred_inverse))
        metrics_reg['precision'].append(precision_score(y_test_raw, y_pred_inverse, average='weighted', zero_division=0))
        metrics_reg['f1'].append(f1_score(y_test_raw, y_pred_inverse, average='weighted', zero_division=0))

        # Confusion matrix
        cm = confusion_matrix(y_test_raw, y_pred_inverse, labels=unique_labels)
        total_conf_matrix += cm

        # Combine model coefficients into a DataFrame
        n_features = X_train_rep.shape[1]
        param_names = [f'param_{i}' for i in range(n_features)]
        model_params_df = pd.DataFrame(model_params_list, columns=param_names)
    
        r2 = r2_score(y_test_transformed, y_pred_transformed)
        mse = mean_squared_error(y_test_transformed, y_pred_transformed)
    
        # Compute classification metric (accuracy)
        acc = accuracy_score(y_test_raw, y_pred_inverse)

        #print(y_pred_transformed)
        # Store results
        if alpha==0:
            results.append([r20, mse0, acc])
        else:
            results.append([r2, mse, acc])

    return metrics_reg, total_conf_matrix, unique_labels, model_params_df, results

In [9]:
def per_class_metrics(conf_matrix, class_names=None):

    conf_matrix = np.asarray(conf_matrix)

    if conf_matrix.ndim != 2:
        raise ValueError("The confusion matrix must be two-dimensional.")

    if conf_matrix.shape[0] != conf_matrix.shape[1]:
        raise ValueError("The confusion matrix must be square.")

    n_classes = conf_matrix.shape[0]

    if class_names is None:
        class_names = [f"Class {i}" for i in range(n_classes)]

    if len(class_names) != n_classes:
        raise ValueError(
            "The number of class names must match the size of the "
            "confusion matrix."
        )

    true_positive = np.diag(conf_matrix)
    predicted_total = conf_matrix.sum(axis=0)
    actual_total = conf_matrix.sum(axis=1)

    precision = np.divide(
        true_positive,
        predicted_total,
        out=np.zeros(n_classes, dtype=float),
        where=predicted_total != 0,
    )

    recall = np.divide(
        true_positive,
        actual_total,
        out=np.zeros(n_classes, dtype=float),
        where=actual_total != 0,
    )

    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(n_classes, dtype=float),
        where=(precision + recall) != 0,
    )

    return pd.DataFrame(
        {
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "Support": actual_total,
        },
        index=class_names,
    )

3 classes (0-3, 4-6, 7-10)

In [10]:
# Convert labels
y_levels = convert_to_levels(y)

metrics_reg, conf_matrix_reg, labels_3lvl, model_params_df, results = run_multiple_evaluations_with_ridge(
    X_freq, y_levels, N0, c, N=100, alpha = 0.05
)
N=100;
summarize_metrics(metrics_reg, "Ridge Transformed Replicator Regression")


Ridge Transformed Replicator Regression Average Metrics over 100 runs:
Accuracy  : 0.8300 ± 0.0354
Precision : 0.8485 ± 0.0367
F1        : 0.8361 ± 0.0344


In [11]:
conf_matrix = conf_matrix_reg

metrics = per_class_metrics(
    conf_matrix,
    class_names=["Healthy", "Intermediate", "BV"],
)

print(metrics.round(4))

              Precision  Recall  F1-score  Support
Healthy          0.9164  0.8683    0.8917     5010
Intermediate     0.4342  0.5393    0.4811      979
BV               0.8668  0.8786    0.8727     1911


Train on 3 and test on 2 classes (0-6, 7-10)

In [12]:
def run_multiple_evaluations_with_ridge_grouped_eval2(
    X, y, N0=None, c=None, N=100, test_size=0.2, alpha=1.0, random_state_seed=42
):

    unique_labels = sorted(y.unique())
    metrics_reg = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    model_params_list = []
    results = []

    for i in range(N):
        X_train, X_test, y_train_raw, y_test_raw = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # Transform y for regression
        y_train_transformed = transform_y(y_train_raw / 10, N0, c)
        y_test_transformed = transform_y(y_test_raw / 10, N0, c)

        # Replicator features
        X_train_rep = construct_replicator_features(X_train)
        X_test_rep = construct_replicator_features(X_test)

        # Fit Ridge model
        if alpha == 0:
            model = sm.OLS(y_train_transformed, X_train_rep).fit()
            r20 = model.rsquared
            mse0 = np.mean(model.resid ** 2)
        else:
            model = Ridge(alpha=alpha, fit_intercept=False)
            model.fit(X_train_rep, y_train_transformed)
            model_params_list.append(model.coef_)

        # Predict and inverse-transform
        y_pred_transformed = model.predict(X_test_rep)
        y_pred_inverse = []

        if N0 is not None and c is not None:
            N_max = 10
            for pred in y_pred_transformed:
                pred = max(min(pred, 100), -100)
                N_pred = N_max + c - (N_max - N0 + c) / np.exp(pred)
                closest_label = min(unique_labels, key=lambda x: abs(x - 10 * N_pred))
                y_pred_inverse.append(closest_label)
        else:
            y_pred_inverse = np.round(10 * y_pred_transformed).astype(int)
            y_pred_inverse = np.clip(y_pred_inverse, min(unique_labels), max(unique_labels))

        y_pred_inverse = np.array(y_pred_inverse)

        # === Evaluation with 3 levels (for reference) ===
        cm_3lvl = confusion_matrix(y_test_raw, y_pred_inverse, labels=unique_labels)
        total_conf_matrix += cm_3lvl

        # === Convert both true & predicted labels to 2-level groups ===
        def group_to_2levels(y_val):
            mapping = {15: 0, 50: 0, 85: 1}
            #mapping = {10:0, 20:0, 30:0, 40:0, 50:0, 60:0, 70:1, 80:1, 90:1, 100:1}
            return np.array([mapping[v] for v in y_val])

        y_test_grouped = group_to_2levels(y_test_raw)
        y_pred_grouped = group_to_2levels(y_pred_inverse)

        # === Compute metrics on grouped (2-level) labels ===
        acc = accuracy_score(y_test_grouped, y_pred_grouped)
        prec = precision_score(y_test_grouped, y_pred_grouped, average='weighted', zero_division=0)
        f1 = f1_score(y_test_grouped, y_pred_grouped, average='weighted', zero_division=0)

        
        metrics_reg['accuracy'].append(acc)
        metrics_reg['precision'].append(prec)
        metrics_reg['f1'].append(f1)
        
        # Regression metrics
        r2 = r2_score(y_test_transformed, y_pred_transformed)
        mse = mean_squared_error(y_test_transformed, y_pred_transformed)

        if alpha == 0:
            results.append([r20, mse0, acc])
        else:
            results.append([r2, mse, acc])

    # Combine model coefficients into a DataFrame
    if model_params_list:
        n_features = X_train_rep.shape[1]
        param_names = [f'param_{i}' for i in range(n_features)]
        model_params_df = pd.DataFrame(model_params_list, columns=param_names)
    else:
        model_params_df = pd.DataFrame()

    # Group the first two classes of the 3-level confusion matrix
    grouped_conf_matrix = np.array([
        [
            total_conf_matrix[:2, :2].sum(),  # true classes 1/2, predicted classes 1/2
            total_conf_matrix[:2, 2].sum(),   # true classes 1/2, predicted class 3
        ],
        [
            total_conf_matrix[2, :2].sum(),   # true class 3, predicted classes 1/2
            total_conf_matrix[2, 2],          # true class 3, predicted class 3
        ],
    ])

    return metrics_reg, grouped_conf_matrix, unique_labels, model_params_df, results


In [13]:
metrics_reg, conf_matrix_reg, labels_3lvl, model_params_df, results = run_multiple_evaluations_with_ridge_grouped_eval2(
    X_freq, y_levels, N0, c, N=100, alpha = 0.05
)

summarize_metrics(metrics_reg, "Ridge Transformed Replicator Regression (Grouped 2-level Eval)")


Ridge Transformed Replicator Regression (Grouped 2-level Eval) Average Metrics over 100 runs:
Accuracy  : 0.9380 ± 0.0250
Precision : 0.9397 ± 0.0242
F1        : 0.9381 ± 0.0248


In [14]:
conf_matrix = conf_matrix_reg

metrics = per_class_metrics(
    conf_matrix,
    class_names=["Healthy+Intermediate", "BV"],
)

print(metrics.round(4))

                      Precision  Recall  F1-score  Support
Healthy+Intermediate     0.9611  0.9569    0.9590     5989
BV                       0.8668  0.8786    0.8727     1911


In [15]:
def run_multiple_evaluations_with_ridge_grouped_eval2(
    X,
    y,
    N0=None,
    c=None,
    N=100,
    test_size=0.2,
    alpha=1.0,
    random_state_seed=42,
):
    

    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    unique_labels = sorted(y.unique())

    metrics_reg = {
        "accuracy": [],
        "precision": [],
        "f1": [],
        "auc": [],
    }

    total_conf_matrix = np.zeros(
        (len(unique_labels), len(unique_labels)),
        dtype=int,
    )

    model_params_list = []
    results = []

    def group_to_2levels(y_values):
        mapping = {
            15: 0,
            50: 0,
            85: 1,
        }

        unknown_labels = set(np.asarray(y_values)) - set(mapping)

        if unknown_labels:
            raise ValueError(
                "Unexpected labels found when grouping outcomes: "
                f"{sorted(unknown_labels)}"
            )

        return np.array(
            [mapping[value] for value in y_values],
            dtype=int,
        )

    for i in range(N):
        (
            X_train,
            X_test,
            y_train_raw,
            y_test_raw,
        ) = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=random_state_seed + i,
            stratify=y,
        )

        # Transform the response for regression
        y_train_transformed = transform_y(
            y_train_raw / 10,
            N0,
            c,
        )

        y_test_transformed = transform_y(
            y_test_raw / 10,
            N0,
            c,
        )

        # Construct replicator features
        X_train_rep = construct_replicator_features(X_train)
        X_test_rep = construct_replicator_features(X_test)

        # Fit either ordinary least squares or ridge regression
        if alpha == 0:
            model = sm.OLS(
                y_train_transformed,
                X_train_rep,
            ).fit()

            train_r2 = model.rsquared
            train_mse = np.mean(model.resid**2)

            model_params_list.append(
                np.asarray(model.params)
            )

        else:
            model = Ridge(
                alpha=alpha,
                fit_intercept=False,
            )

            model.fit(
                X_train_rep,
                y_train_transformed,
            )

            model_params_list.append(
                np.asarray(model.coef_)
            )

        # Continuous predictions on the transformed scale
        y_pred_transformed = np.asarray(
            model.predict(X_test_rep)
        )

        # Convert continuous predictions back to the nearest original label
        if N0 is not None and c is not None:
            N_max = 10

            clipped_predictions = np.clip(
                y_pred_transformed,
                -100,
                100,
            )

            predicted_nugent = (
                N_max
                + c
                - (N_max - N0 + c)
                / np.exp(clipped_predictions)
            )

            predicted_nugent = 10 * predicted_nugent

            y_pred_inverse = np.array(
                [
                    min(
                        unique_labels,
                        key=lambda label: abs(
                            label - prediction
                        ),
                    )
                    for prediction in predicted_nugent
                ]
            )

        else:
            y_pred_inverse = np.round(
                10 * y_pred_transformed
            ).astype(int)

            y_pred_inverse = np.clip(
                y_pred_inverse,
                min(unique_labels),
                max(unique_labels),
            )

        # Three-level confusion matrix
        cm_3lvl = confusion_matrix(
            y_test_raw,
            y_pred_inverse,
            labels=unique_labels,
        )

        total_conf_matrix += cm_3lvl

        # Group true and predicted labels into two classes
        y_test_grouped = group_to_2levels(
            y_test_raw
        )

        y_pred_grouped = group_to_2levels(
            y_pred_inverse
        )

        # Grouped classification metrics
        acc = accuracy_score(
            y_test_grouped,
            y_pred_grouped,
        )

        prec = precision_score(
            y_test_grouped,
            y_pred_grouped,
            average="weighted",
            zero_division=0,
        )

        f1 = f1_score(
            y_test_grouped,
            y_pred_grouped,
            average="weighted",
            zero_division=0,
        )

        # AUROC uses the continuous prediction score
        auc = roc_auc_score(
            y_test_grouped,
            y_pred_transformed,
        )

        metrics_reg["accuracy"].append(acc)
        metrics_reg["precision"].append(prec)
        metrics_reg["f1"].append(f1)
        metrics_reg["auc"].append(auc)

        # Regression metrics on the transformed scale
        test_r2 = r2_score(
            y_test_transformed,
            y_pred_transformed,
        )

        test_mse = mean_squared_error(
            y_test_transformed,
            y_pred_transformed,
        )

        if alpha == 0:
            results.append(
                [
                    train_r2,
                    train_mse,
                    acc,
                ]
            )
        else:
            results.append(
                [
                    test_r2,
                    test_mse,
                    acc,
                ]
            )

    # Combine coefficients from all fitted models
    if model_params_list:
        n_features = len(model_params_list[0])

        param_names = [
            f"param_{i}"
            for i in range(n_features)
        ]

        model_params_df = pd.DataFrame(
            model_params_list,
            columns=param_names,
        )

    else:
        model_params_df = pd.DataFrame()

    return (
        metrics_reg,
        total_conf_matrix,
        unique_labels,
        model_params_df,
        results,
    )

metrics_reg, conf_matrix_reg, labels_3lvl, model_params_df, results = run_multiple_evaluations_with_ridge_grouped_eval2(
    X_freq, y_levels, N0, c, N=100, alpha = 0.05
)

print(
    "Mean AUROC:",
    np.mean(metrics_reg["auc"]),
)

print(
    "AUROC standard deviation:",
    np.std(metrics_reg["auc"]),
)


Mean AUROC: 0.9622017543859649
AUROC standard deviation: 0.0189209916305778


# Performance by ethnicity (requires that we use the dataset differently)

In [16]:
# Extract the ethnicity column
meta_col = data.iloc[:, 2]  

# Extract Nugent Score (at index 3)
y = data.iloc[:, 3].astype(int)

# Features start from column index 4 onward
X_raw = X_freq

# Convert to frequencies per row
# X_freq = X_raw.div(X_raw.sum(axis=1), axis=0)

# X_freq = X_freq[sorted(X_freq.columns)].iloc[:,1:]
X_freq.insert(0, meta_col.name, meta_col)

In [17]:
def run_multiple_evaluations_with_metrics_ridge_by_race(
    X, y, N0=None, c=None, alpha=1.0, N=100, test_size=0.2, random_state_seed=42
):
    """
    Evaluate Ridge Regression (with transformation) and Random Forest
    over multiple random splits and across race groups.
    """

    # Extract race and features
    race_column = X.iloc[:, 0]
    X_features = X.iloc[:, 1:]
    unique_labels = sorted(y.unique())
    unique_races = sorted(race_column.unique())
    num_classes = len(unique_labels)
    all_groups = unique_races + ['overall']

    # Initialize confusion matrices and metrics
    conf_matrices_ridge = {g: np.zeros((num_classes, num_classes), dtype=int) for g in all_groups}
    conf_matrices_rf = {g: np.zeros((num_classes, num_classes), dtype=int) for g in all_groups}
    metrics_ridge = {g: {'accuracy': [], 'precision': [], 'f1': []} for g in all_groups}
    metrics_rf = {g: {'accuracy': [], 'precision': [], 'f1': []} for g in all_groups}

    model_coefs_list = []  # store ridge coefficients for inspection

    for i in range(N):
        train_idx, test_idx = train_test_split(
            X.index, test_size=test_size, random_state=random_state_seed + i
        )
        
        # Subset using indices
        X_train_full = X.loc[train_idx]
        X_test_full = X.loc[test_idx]
        y_train = y.loc[train_idx]
        y_test = y.loc[test_idx]

        race_train = X_train_full.iloc[:, 0]
        race_test = X_test_full.iloc[:, 0]
        X_train = X_train_full.iloc[:, 1:]
        X_test = X_test_full.iloc[:, 1:]

        # Transform response
        y_train_transformed = transform_y(y_train / 10, N0, c)
        y_test_transformed = transform_y(y_test / 10, N0, c)

        # Replicator features
        X_train_rep = construct_replicator_features(X_train)
        X_test_rep = construct_replicator_features(X_test)

        # Fit Ridge regression
        ridge_model = Ridge(alpha=alpha, fit_intercept=False)
        ridge_model.fit(X_train_rep, y_train_transformed)
        model_coefs_list.append(ridge_model.coef_)
        
        # Predict and inverse-transform
        y_pred_transformed = ridge_model.predict(X_test_rep)

        y_pred_inverse = []
        N_max = 10
        for pred in y_pred_transformed:
            pred = np.clip(pred, -100, 100)
            N_pred = N_max + c - (N_max - N0 + c) / np.exp(pred)
            closest_label = min(unique_labels, key=lambda x: abs(x - 10 * N_pred))
            y_pred_inverse.append(closest_label)
        y_pred_inverse = np.array(y_pred_inverse)

        # Random Forest baseline
        rf = RandomForestClassifier(n_estimators=200, random_state=random_state_seed + i)
        rf.fit(X_train, y_train)
        y_pred_rf = rf.predict(X_test)

        # Compute metrics overall
        cm_ridge = confusion_matrix(y_test, y_pred_inverse, labels=unique_labels)
        cm_rf = confusion_matrix(y_test, y_pred_rf, labels=unique_labels)
        conf_matrices_ridge['overall'] += cm_ridge
        conf_matrices_rf['overall'] += cm_rf

        for metrics_dict, y_pred in [
            (metrics_ridge['overall'], y_pred_inverse),
            (metrics_rf['overall'], y_pred_rf),
        ]:
            metrics_dict['accuracy'].append(accuracy_score(y_test, y_pred))
            metrics_dict['precision'].append(precision_score(y_test, y_pred, average='weighted', zero_division=0))
            metrics_dict['f1'].append(f1_score(y_test, y_pred, average='weighted', zero_division=0))

        # Per-race metrics
        for race in unique_races:
            race_mask = race_test == race
            if not race_mask.any():
                continue

            y_test_race = y_test[race_mask]
            y_pred_ridge_race = y_pred_inverse[race_mask]
            y_pred_rf_race = y_pred_rf[race_mask]

            cm_ridge_race = confusion_matrix(y_test_race, y_pred_ridge_race, labels=unique_labels)
            cm_rf_race = confusion_matrix(y_test_race, y_pred_rf_race, labels=unique_labels)
            conf_matrices_ridge[race] += cm_ridge_race
            conf_matrices_rf[race] += cm_rf_race

            for metrics_dict, y_true, y_pred in [
                (metrics_ridge[race], y_test_race, y_pred_ridge_race),
                (metrics_rf[race], y_test_race, y_pred_rf_race),
            ]:
                metrics_dict['accuracy'].append(accuracy_score(y_true, y_pred))
                metrics_dict['precision'].append(precision_score(y_true, y_pred, average='weighted', zero_division=0))
                metrics_dict['f1'].append(f1_score(y_true, y_pred, average='weighted', zero_division=0))

    # Average metrics
    def summarize_metrics(metrics_dict):
        summary = {}
        for group in all_groups:
            summary[group] = {}
            for m in ['accuracy', 'precision', 'f1']:
                vals = metrics_dict[group][m]
                if vals:
                    summary[group][m] = {'mean': np.mean(vals), 'std': np.std(vals), 'n': len(vals)}
                else:
                    summary[group][m] = {'mean': np.nan, 'std': np.nan, 'n': 0}
        return summary

    stats_ridge = summarize_metrics(metrics_ridge)
    stats_rf = summarize_metrics(metrics_rf)

    # Collect model coefficients (overall)
    feature_names = X_train_rep.columns if hasattr(X_train_rep, 'columns') else [f"feature_{i}" for i in range(X_train_rep.shape[1])]
    model_coefs_df = pd.DataFrame(model_coefs_list, columns=feature_names)

    return (
        conf_matrices_ridge, conf_matrices_rf, unique_labels, unique_races,
        stats_ridge, stats_rf, model_coefs_df
    )




def display_race_performance_results(stats_reg, stats_rf, unique_races):
    """Display performance results in the requested format"""
    
    # Display group-wise performance for Replicator Regression
    print("Replicator Regression - Group-wise Performance (mean ± std):")
    for race in unique_races:
        acc = stats_reg[race]['accuracy']
        prec = stats_reg[race]['precision']
        f1 = stats_reg[race]['f1']
        
        if not np.isnan(acc['mean']):
            print(f"Group {race}: Accuracy={acc['mean']:.3f}±{acc['std']:.3f}, "
                  f"Precision={prec['mean']:.3f}±{prec['std']:.3f}, "
                  f"F1={f1['mean']:.3f}±{f1['std']:.3f}")
        else:
            print(f"Group {race}: No data available")
    
    print()
    
    # Display group-wise performance for Random Forest
    print("Random Forest - Group-wise Performance (mean ± std):")
    for race in unique_races:
        acc = stats_rf[race]['accuracy']
        prec = stats_rf[race]['precision']
        f1 = stats_rf[race]['f1']
        
        if not np.isnan(acc['mean']):
            print(f"Group {race}: Accuracy={acc['mean']:.3f}±{acc['std']:.3f}, "
                  f"Precision={prec['mean']:.3f}±{prec['std']:.3f}, "
                  f"F1={f1['mean']:.3f}±{f1['std']:.3f}")
        else:
            print(f"Group {race}: No data available")
    
    print()
    
    # Display overall performance for Regression
    print("Regression Overall Average Metrics over 100 runs:")
    overall_reg = stats_reg['overall']
    print(f"Accuracy  : {overall_reg['accuracy']['mean']:.4f} ± {overall_reg['accuracy']['std']:.4f}")
    print(f"Precision : {overall_reg['precision']['mean']:.4f} ± {overall_reg['precision']['std']:.4f}")
    print(f"F1        : {overall_reg['f1']['mean']:.4f} ± {overall_reg['f1']['std']:.4f}")
    
    print()
    
    # Display overall performance for Random Forest
    print("Random Forest Overall Average Metrics over 100 runs:")
    overall_rf = stats_rf['overall']
    print(f"Accuracy  : {overall_rf['accuracy']['mean']:.4f} ± {overall_rf['accuracy']['std']:.4f}")
    print(f"Precision : {overall_rf['precision']['mean']:.4f} ± {overall_rf['precision']['std']:.4f}")
    print(f"F1        : {overall_rf['f1']['mean']:.4f} ± {overall_rf['f1']['std']:.4f}")


100 Monte Carlo cross validation with 80/20 train/test split

In [18]:
def convert_to_levels2(y):
    return pd.cut(
        y,
        bins=[-0.1, 6, 10],  # 2 categories
        labels=[30, 85]     # Midpoints
    ).astype(int)

N = 100
N0 = 4.5
c=4
y_levels = convert_to_levels2(y)


conf_matrices_ridge, conf_matrices_rf, unique_labels, unique_races, stats_ridge, stats_rf, model_coefs_df = run_multiple_evaluations_with_metrics_ridge_by_race(
    X_freq, 
    y_levels, 
    N0=N0, 
    c=c, 
    alpha = 0.05,     
    N=N, 
    test_size=0.2
)

display_race_performance_results(stats_ridge, stats_rf, unique_races)


Replicator Regression - Group-wise Performance (mean ± std):
Group Asian: Accuracy=0.928±0.059, Precision=0.948±0.052, F1=0.933±0.056
Group Black: Accuracy=0.945±0.049, Precision=0.948±0.048, F1=0.945±0.049
Group Hispanic: Accuracy=0.886±0.067, Precision=0.899±0.062, F1=0.887±0.066
Group White: Accuracy=0.971±0.036, Precision=0.982±0.028, F1=0.974±0.033

Random Forest - Group-wise Performance (mean ± std):
Group Asian: Accuracy=0.930±0.052, Precision=0.934±0.057, F1=0.923±0.061
Group Black: Accuracy=0.944±0.046, Precision=0.948±0.044, F1=0.944±0.046
Group Hispanic: Accuracy=0.902±0.062, Precision=0.910±0.060, F1=0.902±0.062
Group White: Accuracy=0.973±0.033, Precision=0.981±0.028, F1=0.975±0.031

Regression Overall Average Metrics over 100 runs:
Accuracy  : 0.9319 ± 0.0268
Precision : 0.9363 ± 0.0251
F1        : 0.9329 ± 0.0263

Random Forest Overall Average Metrics over 100 runs:
Accuracy  : 0.9367 ± 0.0248
Precision : 0.9375 ± 0.0249
F1        : 0.9365 ± 0.0250


# Machine Learning algorithms

Need to import the data again, same as before the ethnicity results

## Random Forest

In [490]:
def run_multiple_evaluations_with_rf_grouped_eval2(
    X, y, N=100, test_size=0.2, random_state_seed=42
):

    unique_labels = sorted(y.unique())
    metrics_rf = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    feature_importances_list = []
    results = []

    for i in range(N):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # === Fit Random Forest ===
        rf = RandomForestClassifier(n_estimators=100, random_state=random_state_seed + i)
        rf.fit(X_train, y_train)
        y_pred_rf = rf.predict(X_test)

        # === Store feature importances ===
        feature_importances_list.append(rf.feature_importances_)


        # === Compute metrics on grouped (2-level) labels ===
        acc = accuracy_score(y_test, y_pred_rf)
        prec = precision_score(y_test, y_pred_rf, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)

        metrics_rf['accuracy'].append(acc)
        metrics_rf['precision'].append(prec)
        metrics_rf['f1'].append(f1)

        # Save metrics per run
        results.append([acc, prec, f1])

    # Combine feature importances into a DataFrame
    if feature_importances_list:
        n_features = X.shape[1]
        feature_names = getattr(X, 'columns', [f'feature_{i}' for i in range(n_features)])
        feature_importances_df = pd.DataFrame(feature_importances_list, columns=feature_names)
    else:
        feature_importances_df = pd.DataFrame()

    return metrics_rf, feature_importances_df, results


3 Nugent score levels

In [491]:
y_levels = convert_to_levels(y)

metrics_reg, feature_importances_df, results = run_multiple_evaluations_with_rf_grouped_eval2(
    X_freq.iloc[:,1:], y_levels, N=100
)

summarize_metrics(metrics_reg, "Random Forest(Grouped 2-level Eval)")


Random Forest(Grouped 2-level Eval) Average Metrics over 10 runs of sample size 50:
Accuracy  : 0.8271 ± 0.0379
Precision : 0.8096 ± 0.0490
F1        : 0.8116 ± 0.0442


2 Nugent score levels

In [492]:
y_levels = convert_to_levels2(y)

metrics_reg, feature_importances_df, results = run_multiple_evaluations_with_rf_grouped_eval2(
    X_freq.iloc[:,1:], y_levels, N=100
)

summarize_metrics(metrics_reg, "Random Forest(Grouped 2-level Eval)")


Random Forest(Grouped 2-level Eval) Average Metrics over 10 runs of sample size 50:
Accuracy  : 0.9373 ± 0.0257
Precision : 0.9383 ± 0.0256
F1        : 0.9372 ± 0.0257


## XGBoost

In [489]:
def run_multiple_evaluations_with_xgb_grouped_eval2(
    X, y, N=100, test_size=0.2, random_state_seed=42
):

    unique_labels = sorted(y.unique())
    metrics_xgb = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    results = []

    for i in range(N):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # === Fit XGBoost ===
        xgb = XGBClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='mlogloss',
            random_state=random_state_seed + i,
            n_jobs=-1
        )
        xgb.fit(X_train, y_train)
        y_pred_xgb = xgb.predict(X_test)

        acc = accuracy_score(y_test, y_pred_xgb)
        prec = precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0)

        metrics_xgb['accuracy'].append(acc)
        metrics_xgb['precision'].append(prec)
        metrics_xgb['f1'].append(f1)

        results.append([acc, prec, f1])

    return metrics_xgb, total_conf_matrix, results

In [418]:
#Requires binary classification
def convert_to_levels3(y):
    return pd.cut(
        y,
        bins=[-0.1, 6, 10],  # 3 categories
        labels=[0, 1]     # Midpoints
    ).astype(int)



y_levels = convert_to_levels3(y)

metrics_reg, conf_matrix_reg, results = run_multiple_evaluations_with_xgb_grouped_eval2(
    X_freq.iloc[:,1:], y_levels, N=100
)

summarize_metrics(metrics_reg, "XGBOOST 2 LEVELS")


XGBOOST 2 LEVELS Average Metrics over 100 runs:
Accuracy  : 0.9276 ± 0.0254
Precision : 0.9292 ± 0.0247
F1        : 0.9273 ± 0.0254


## SVM

In [419]:
def run_multiple_evaluations_with_svm_grouped_eval2(
    X, y, N=100, test_size=0.2, random_state_seed=42
):
    unique_labels = sorted(y.unique())
    metrics_svm = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    results = []

    for i in range(N):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # === Fit SVM ===
        svm = SVC(
            kernel='rbf',        # you can change to 'linear', 'poly', etc.
            C=1.0,
            gamma='scale',
            random_state=random_state_seed + i
        )
        svm.fit(X_train, y_train)
        y_pred_svm = svm.predict(X_test)

        # === Compute metrics ===
        acc = accuracy_score(y_test, y_pred_svm)
        prec = precision_score(y_test, y_pred_svm, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred_svm, average='weighted', zero_division=0)

        metrics_svm['accuracy'].append(acc)
        metrics_svm['precision'].append(prec)
        metrics_svm['f1'].append(f1)

        results.append([acc, prec, f1])

        # Optionally accumulate confusion matrix
        cm = confusion_matrix(y_test, y_pred_svm, labels=unique_labels)
        total_conf_matrix += cm

    # No feature importances for SVM
    feature_importances_df = pd.DataFrame()

    return metrics_svm, total_conf_matrix, unique_labels, feature_importances_df, results


In [420]:
metrics_reg, conf_matrix_reg, labels_3lvl, model_params_df, results = run_multiple_evaluations_with_svm_grouped_eval2(
    X_freq.iloc[:, 1:], y_levels, N=100
)

summarize_metrics(metrics_reg, "SVM 2 LEVELS")


SVM 2 LEVELS Average Metrics over 100 runs:
Accuracy  : 0.9323 ± 0.0261
Precision : 0.9352 ± 0.0246
F1        : 0.9329 ± 0.0257


## KNN

In [421]:
def run_multiple_evaluations_with_knn_grouped_eval2(
    X, y, N=100, test_size=0.2, random_state_seed=42, n_neighbors=5
):
    unique_labels = sorted(y.unique())
    metrics_knn = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    results = []

    for i in range(N):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # === Fit KNN ===
        knn = KNeighborsClassifier(n_neighbors=n_neighbors)
        knn.fit(X_train, y_train)
        y_pred_knn = knn.predict(X_test)

        # === Compute metrics ===
        acc = accuracy_score(y_test, y_pred_knn)
        prec = precision_score(y_test, y_pred_knn, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred_knn, average='weighted', zero_division=0)

        metrics_knn['accuracy'].append(acc)
        metrics_knn['precision'].append(prec)
        metrics_knn['f1'].append(f1)

        results.append([acc, prec, f1])

        # Optionally accumulate confusion matrix
        cm = confusion_matrix(y_test, y_pred_knn, labels=unique_labels)
        total_conf_matrix += cm


    return metrics_knn, total_conf_matrix, results

In [422]:
metrics_reg, conf_matrix_reg, results = run_multiple_evaluations_with_knn_grouped_eval2(
    X_freq.iloc[:,:], y_levels, N=100
)

summarize_metrics(metrics_reg, "SVM 2 LEVELS")


SVM 2 LEVELS Average Metrics over 100 runs:
Accuracy  : 0.9311 ± 0.0254
Precision : 0.9344 ± 0.0236
F1        : 0.9318 ± 0.0248


# Linear Regression fitting

In [17]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    confusion_matrix,
)


def run_multiple_evaluations_linear_quadratic(
    X,
    y,
    N=100,
    test_size=0.2,
    random_state_seed=42,
    C=1.0,
):
    """
    Evaluate a logistic regression model using:

        - all original linear terms
        - all distinct pairwise interaction terms

    For 10 input features, this gives:
        10 linear terms + 45 pairwise terms = 55 predictors

    Squared terms such as x_i**2 are not included.
    """

    unique_labels = sorted(y.unique())

    metrics_model = {
        "accuracy": [],
        "precision": [],
        "f1": [],
    }

    total_conf_matrix = np.zeros(
        (len(unique_labels), len(unique_labels)),
        dtype=int,
    )

    results = []

    for i in range(N):
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=random_state_seed + i,
            stratify=y,
        )

        poly = PolynomialFeatures(
            degree=2,
            interaction_only=True,
            include_bias=False,
        )
        
        X_train_expanded = poly.fit_transform(X_train)
        X_test_expanded = poly.transform(X_test)
        
        model = LogisticRegression(penalty=None, max_iter=5000,
        )
        
        model.fit(X_train_expanded, y_train)
        y_pred = model.predict(X_test_expanded)


        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0,
        )
        f1 = f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0,
        )

        metrics_model["accuracy"].append(acc)
        metrics_model["precision"].append(prec)
        metrics_model["f1"].append(f1)

        results.append([acc, prec, f1])

        cm = confusion_matrix(
            y_test,
            y_pred,
            labels=unique_labels,
        )
        total_conf_matrix += cm

    return metrics_model, total_conf_matrix, results

In [424]:
def convert_to_levels2(y):
    return pd.cut(
        y,
        bins=[-0.1, 6, 10],  # 2 categories
        labels=[0, 1]     # Midpoints
    ).astype(int)

y_levels = convert_to_levels2(y)

metrics_reg, conf_matrix_reg, results_reg = (
    run_multiple_evaluations_linear_quadratic(
        X_freq.iloc[:, 1:],
        y_levels,
        N=100,
    )
)

summarize_metrics(
    metrics_reg,
    "LOGISTIC REGRESSION — LINEAR + PAIRWISE TERMS — 2 LEVELS",
)


LOGISTIC REGRESSION — LINEAR + PAIRWISE TERMS — 2 LEVELS Average Metrics over 100 runs:
Accuracy  : 0.9076 ± 0.0291
Precision : 0.9097 ± 0.0293
F1        : 0.9075 ± 0.0293


Visualizing the coefficients:

In [425]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression


X_model = X_freq.iloc[:, 1:]

# Generate the 10 linear + 45 pairwise interaction terms
poly = PolynomialFeatures(
    degree=2,
    interaction_only=True,
    include_bias=False,
)

X_expanded = poly.fit_transform(X_model)

# Fit logistic regression using the entire dataset
model_full = LogisticRegression(penalty=None,
    max_iter=5000,
)

model_full.fit(X_expanded, y_levels)

# Get the names of the 55 model terms
feature_names = poly.get_feature_names_out(X_model.columns)

# Create a coefficient table
coefficient_table = pd.DataFrame({
    "term": feature_names,
    "coefficient": model_full.coef_[0],
})

# Optional: order exactly as generated:
# first linear terms, then pairwise terms
print(coefficient_table.to_string(index=False))

print("\nIntercept:", model_full.intercept_[0])
print("Class order:", model_full.classes_)

                                          term  coefficient
                            Bacillota_A_368345     0.806593
                                   Bacillota_C   -24.038123
                             Bacillota_I_other    -9.017090
                           Lactobacillus_iners    -9.065917
                       Lactobacillus_crispatus  -233.792519
                        Lactobacillus_jensenii   -77.718641
                         Lactobacillus_gasseri   -24.639520
                                  Bacteroidota   -13.753373
                            Campylobacterota_A    49.611438
                                Fusobacteriota    45.118334
                               Patescibacteria    31.984722
                                Pseudomonadota  -334.334881
                                  Synergistota    12.331739
                Bacillota_A_368345 Bacillota_C    15.677779
          Bacillota_A_368345 Bacillota_I_other   -12.620281
        Bacillota_A_368345 Lactobacillus

# Validation on a different dataset

Need to import the data again, same as before the ethnicity results

In [57]:
def convert_to_levels(y):
    return pd.cut(
        y,
        bins=[-0.1, 3, 6, 10],  # 3 categories
        labels=[15, 50, 85]     # Midpoints
    ).astype(int)
    
y_levels = convert_to_levels(y)

N0=4.5
c=4

results_df, fitted_model, r2 = fit_transformed_ridge_on_full_data(
    X_freq,
    y_levels,
    N0=N0,
    c=c,
    alpha=0.05,
    n_boot=1000,
    random_state=42,
)

In [58]:
import numpy as np
import pandas as pd


TAXON_COLUMNS = [
    "Actinomycetota",
    "Bacillota_A_368345",
    "Bacillota_C",
    "Bacillota_I_other",
    "Lactobacillus_iners",
    "Lactobacillus_crispatus",
    "Lactobacillus_jensenii",
    "Lactobacillus_gasseri",
    "Bacteroidota",
    "Campylobacterota_A",
    "Fusobacteriota",
    "Patescibacteria",
    "Pseudomonadota",
    "Synergistota",
]


def load_external_longitudinal_data(
    filename="ravel_longitudinal_GTDB_with_nugent.csv",
):
    data = pd.read_csv(filename)

    required_columns = (
        ["sampleID", "time", "Nugent_Score"]
        + TAXON_COLUMNS
    )

    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    # Convert relevant columns to numeric
    for column in ["time", "Nugent_Score"] + TAXON_COLUMNS:
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce",
        )

    # Remove samples with missing Nugent scores or abundance values
    data = data.dropna(
        subset=["Nugent_Score"] + TAXON_COLUMNS
    ).copy()

    # Validate Nugent scores
    if not data["Nugent_Score"].between(0, 10).all():
        invalid = data.loc[
            ~data["Nugent_Score"].between(0, 10),
            ["sampleID", "Nugent_Score"],
        ]

        raise ValueError(
            "Nugent scores outside the range 0-10:\n"
            + invalid.to_string(index=False)
        )

    X_percent = data[TAXON_COLUMNS].copy()

    # --------------------------------------------------------
    # Detect whether abundances are percentages or frequencies
    # --------------------------------------------------------
    original_row_sums = X_percent.sum(axis=1)

    if original_row_sums.median() <= 2:
        # Data are probably frequencies already
        X_percent = X_percent * 100.0

    # --------------------------------------------------------
    # Remove impossible rows
    # --------------------------------------------------------
    row_sums_before = X_percent.sum(axis=1)

    valid_rows = row_sums_before > 0

    if not valid_rows.all():
        print(
            "Removing samples with total mapped abundance equal to zero:"
        )
        print(
            data.loc[
                ~valid_rows,
                "sampleID",
            ].tolist()
        )

        data = data.loc[valid_rows].copy()
        X_percent = X_percent.loc[valid_rows].copy()
        row_sums_before = row_sums_before.loc[valid_rows]

    # --------------------------------------------------------
    # Renormalize mapped taxa to exactly 100%
    # --------------------------------------------------------
    X_percent = (
        X_percent
        .div(X_percent.sum(axis=1), axis=0)
        .mul(100.0)
    )

    # --------------------------------------------------------
    # Convert percentages to frequencies for the fitted model
    # --------------------------------------------------------
    X_external = X_percent / 100.0

    y_external = data["Nugent_Score"].astype(int)

    # Reset all indices together
    data = data.reset_index(drop=True)
    X_percent = X_percent.reset_index(drop=True)
    X_external = X_external.reset_index(drop=True)
    y_external = y_external.reset_index(drop=True)

    print(f"External samples retained: {len(data)}")

    print("\nMapped abundance sums before renormalization:")
    print(row_sums_before.describe())

    print("\nPercentage sums after renormalization:")
    print(X_percent.sum(axis=1).describe())

    print("\nFrequency sums supplied to the model:")
    print(X_external.sum(axis=1).describe())

    return data, X_external, y_external

In [59]:
external_data, X_external, y_external = (
    load_external_longitudinal_data(
        "ravel_longitudinal_GTDB_with_nugent.csv"
    )
)

External samples retained: 1488

Mapped abundance sums before renormalization:
count    1488.000000
mean       99.881466
std         0.184687
min        99.038050
25%        99.851214
50%        99.961427
75%       100.000000
max       100.000000
dtype: float64

Percentage sums after renormalization:
count    1.488000e+03
mean     1.000000e+02
std      1.406195e-14
min      1.000000e+02
25%      1.000000e+02
50%      1.000000e+02
75%      1.000000e+02
max      1.000000e+02
dtype: float64

Frequency sums supplied to the model:
count    1.488000e+03
mean     1.000000e+00
std      1.465792e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


In [60]:
external_data

,sampleID,time,Nugent_Score,Actinomycetota,Bacillota_A_368345,Bacillota_C,Bacillota_I_other,Lactobacillus_iners,Lactobacillus_crispatus,Lactobacillus_jensenii,Lactobacillus_gasseri,Bacteroidota,Campylobacterota_A,Fusobacteriota,Patescibacteria,Pseudomonadota,Synergistota
0,s3.w1d1,8,1,5.339806,0.000000,0.000000,0.000000,67.961165,4.854369,21.844660,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0
1,s3.w1d2,9,4,1.524390,0.000000,0.000000,0.609756,67.682927,7.926829,22.256098,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0
2,s3.w1d3,10,1,2.371542,0.000000,0.000000,0.000000,68.379447,13.043478,16.205534,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0
3,s3.w1d5,12,6,23.076923,0.641026,3.205128,19.871795,50.641026,0.641026,1.282051,0.0,0.641026,0.0,0.000000,0.0,0.000000,0.0
4,s3.w1d6,13,6,60.305344,0.000000,0.763359,12.213740,23.664122,0.000000,1.526718,0.0,1.526718,0.0,0.000000,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,s135.w10d3,73,2,4.075949,1.000000,0.329114,2.075949,82.556962,7.797468,1.582278,0.0,0.329114,0.0,0.063291,0.0,0.000000,0.0
1484,s135.w10d4,74,2,4.383484,0.622172,0.622172,2.333145,82.409502,7.904412,1.555430,0.0,0.098982,0.0,0.056561,0.0,0.000000,0.0
1485,s135.w10d5,75,3,13.169726,1.517422,0.412139,3.353316,78.287748,1.948295,0.618209,0.0,0.355939,0.0,0.187336,0.0,0.074934,0.0
1486,s135.w10d7,77,2,4.318593,0.905094,0.181019,1.073183,66.873545,22.976468,3.116111,0.0,0.232739,0.0,0.142229,0.0,0.012930,0.0


In [61]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    confusion_matrix,
    r2_score,
    mean_squared_error,
)


def validate_external_3levels(
    X_external,
    nugent_external,
    fitted_model,
    N0=None,
    c=None,
    N=100,
    subset_size=None,
    random_state_seed=42,
):
    

    X_external = X_external.reset_index(drop=True)
    nugent_external = pd.Series(
        nugent_external
    ).reset_index(drop=True)

    if subset_size is None:
        subset_size = len(X_external)

    if subset_size > len(X_external):
        raise ValueError(
            "subset_size cannot exceed the number of external samples."
        )

    # Convert exact Nugent scores to the labels used for training
    y_external = pd.Series(
        np.select(
            [
                nugent_external <= 3,
                nugent_external <= 6,
            ],
            [
                15,
                50,
            ],
            default=85,
        ),
        index=X_external.index,
    ).astype(int)

    unique_labels = [15, 50, 85]

    metrics_reg = {
        "accuracy": [],
        "precision": [],
        "f1": [],
    }

    total_conf_matrix = np.zeros(
        (3, 3),
        dtype=int,
    )

    results = []

    for i in range(N):

        # Random external subset
        subset_idx = X_external.sample(
            n=subset_size,
            replace=False,
            random_state=random_state_seed + i,
        ).index

        X_test = X_external.loc[subset_idx]
        y_test_raw = y_external.loc[subset_idx]

        # Transform true labels for regression metrics
        y_test_transformed = transform_y(
            y_test_raw / 10,
            N0,
            c,
        )

        # Construct the same features used during training
        X_test_rep = construct_replicator_features(
            X_test
        )

        # Predict using the fixed model
        y_pred_transformed = fitted_model.predict(
            X_test_rep
        )

        # Convert predictions back to the label scale
        if N0 is not None and c is not None:

            y_pred_inverse = []

            for pred in y_pred_transformed:

                pred = np.clip(
                    pred,
                    -100,
                    100,
                )

                N_max = 10

                N_pred = (
                    N_max
                    + c
                    - (N_max - N0 + c)
                    / np.exp(pred)
                )

                # Convert from divided-by-10 scale back to label scale
                N_pred = 10 * N_pred

                closest_label = min(
                    unique_labels,
                    key=lambda label: abs(label - N_pred),
                )

                y_pred_inverse.append(
                    closest_label
                )

            y_pred_inverse = np.array(
                y_pred_inverse
            )

        else:

            y_pred_inverse = np.round(
                10 * y_pred_transformed
            )

            y_pred_inverse = np.array(
                [
                    min(
                        unique_labels,
                        key=lambda label: abs(label - pred),
                    )
                    for pred in y_pred_inverse
                ]
            )

        # Classification metrics
        acc = accuracy_score(
            y_test_raw,
            y_pred_inverse,
        )

        precision = precision_score(
            y_test_raw,
            y_pred_inverse,
            average="weighted",
            zero_division=0,
        )

        f1 = f1_score(
            y_test_raw,
            y_pred_inverse,
            average="weighted",
            zero_division=0,
        )

        metrics_reg["accuracy"].append(acc)
        metrics_reg["precision"].append(precision)
        metrics_reg["f1"].append(f1)

        # Confusion matrix
        cm = confusion_matrix(
            y_test_raw,
            y_pred_inverse,
            labels=unique_labels,
        )

        total_conf_matrix += cm

        # Regression metrics in transformed space
        r2 = r2_score(
            y_test_transformed,
            y_pred_transformed,
        )

        mse = mean_squared_error(
            y_test_transformed,
            y_pred_transformed,
        )

        results.append(
            [r2, mse, acc]
        )

    results_df = pd.DataFrame(
        results,
        columns=[
            "r2",
            "mse",
            "accuracy",
        ],
    )

    return (
        metrics_reg,
        total_conf_matrix,
        unique_labels,
        results_df,
    )

In [62]:
metrics_external_3level, cm_external_3level, labels_3level, results_external_3level = (
    validate_external_3levels(
        X_external=X_external,
        nugent_external=y_external,
        fitted_model=fitted_model,
        N0=N0,
        c=c,
        N=10,
        subset_size=50,
        random_state_seed=42,
    )
)

In [63]:
def validate_external_2levels(
    X_external,
    nugent_external,
    fitted_model,
    N0=None,
    c=None,
    N=100,
    subset_size=None,
    random_state_seed=42,
):


    X_external = X_external.reset_index(drop=True)
    nugent_external = pd.Series(
        nugent_external
    ).reset_index(drop=True)

    if subset_size is None:
        subset_size = len(X_external)

    if subset_size > len(X_external):
        raise ValueError(
            "subset_size cannot exceed the number of external samples."
        )

    # Three-level labels used by the fitted model
    y_external = pd.Series(
        np.select(
            [
                nugent_external <= 3,
                nugent_external <= 6,
            ],
            [
                15,
                50,
            ],
            default=85,
        ),
        index=X_external.index,
    ).astype(int)

    model_labels = [15, 50, 85]
    grouped_labels = [0, 1]

    metrics_reg = {
        "accuracy": [],
        "precision": [],
        "f1": [],
    }

    total_conf_matrix = np.zeros(
        (2, 2),
        dtype=int,
    )

    results = []

    for i in range(N):

        subset_idx = X_external.sample(
            n=subset_size,
            replace=False,
            random_state=random_state_seed + i,
        ).index

        X_test = X_external.loc[subset_idx]
        y_test_raw = y_external.loc[subset_idx]

        # Transform true three-level labels because this is how
        # the fixed model was trained
        y_test_transformed = transform_y(
            y_test_raw / 10,
            N0,
            c,
        )

        X_test_rep = construct_replicator_features(
            X_test
        )

        # Predict using the fixed model
        y_pred_transformed = fitted_model.predict(
            X_test_rep
        )

        # Inverse-transform to three-level labels
        if N0 is not None and c is not None:

            y_pred_inverse = []

            for pred in y_pred_transformed:

                pred = np.clip(
                    pred,
                    -100,
                    100,
                )

                N_max = 10

                N_pred = (
                    N_max
                    + c
                    - (N_max - N0 + c)
                    / np.exp(pred)
                )

                N_pred = 10 * N_pred

                closest_label = min(
                    model_labels,
                    key=lambda label: abs(label - N_pred),
                )

                y_pred_inverse.append(
                    closest_label
                )

            y_pred_inverse = np.array(
                y_pred_inverse
            )

        else:

            continuous_predictions = np.round(
                10 * y_pred_transformed
            )

            y_pred_inverse = np.array(
                [
                    min(
                        model_labels,
                        key=lambda label: abs(label - pred),
                    )
                    for pred in continuous_predictions
                ]
            )

        # Group both true and predicted labels
        mapping = {
            15: 0,
            50: 0,
            85: 1,
        }

        y_test_grouped = np.array(
            [
                mapping[value]
                for value in y_test_raw
            ]
        )

        y_pred_grouped = np.array(
            [
                mapping[value]
                for value in y_pred_inverse
            ]
        )

        # Grouped classification metrics
        acc = accuracy_score(
            y_test_grouped,
            y_pred_grouped,
        )

        precision = precision_score(
            y_test_grouped,
            y_pred_grouped,
            average="weighted",
            zero_division=0,
        )

        f1 = f1_score(
            y_test_grouped,
            y_pred_grouped,
            average="weighted",
            zero_division=0,
        )

        metrics_reg["accuracy"].append(acc)
        metrics_reg["precision"].append(precision)
        metrics_reg["f1"].append(f1)

        # Actual two-class confusion matrix
        cm = confusion_matrix(
            y_test_grouped,
            y_pred_grouped,
            labels=grouped_labels,
        )

        total_conf_matrix += cm

        # Regression metrics remain based on the original model target
        r2 = r2_score(
            y_test_transformed,
            y_pred_transformed,
        )

        mse = mean_squared_error(
            y_test_transformed,
            y_pred_transformed,
        )

        results.append(
            [r2, mse, acc]
        )

    results_df = pd.DataFrame(
        results,
        columns=[
            "r2",
            "mse",
            "accuracy",
        ],
    )

    return (
        metrics_reg,
        total_conf_matrix,
        grouped_labels,
        results_df,
    )

In [64]:
metrics_external_2level, cm_external_2level, labels_2level, results_external_2level = (
    validate_external_2levels(
        X_external=X_external,
        nugent_external=y_external,
        fitted_model=fitted_model,
        N0=N0,
        c=c,
        N=10,
        subset_size=50,
        random_state_seed=42,
    )
)

In [65]:
#Summarize metrics
def summarize_metrics(metrics_dict, name):
    print(f"\n{name} Average Metrics over 10 runs of sample size 50:")
    for key, values in metrics_dict.items():
        print(f"{key.capitalize():<10}: {np.mean(values):.4f} ± {np.std(values):.4f}")

In [66]:
summarize_metrics(
    metrics_external_3level,
    "External validation — 3 levels",
)

summarize_metrics(
    metrics_external_2level,
    "External validation — 2 levels",
)


External validation — 3 levels Average Metrics over 10 runs of sample size 50:
Accuracy  : 0.7720 ± 0.0337
Precision : 0.7750 ± 0.0455
F1        : 0.7658 ± 0.0379

External validation — 2 levels Average Metrics over 10 runs of sample size 50:
Accuracy  : 0.8520 ± 0.0349
Precision : 0.8658 ± 0.0328
F1        : 0.8530 ± 0.0348
